# False Positive / Lookalike Analysis
**Goal:** Inspect "NO OIL" (0) samples that the model incorrectly predicted as "OIL" (1).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!pip install rasterio
%cd /content
!rm -rf Deep-Learning-Oil-Slick-Detection
!git clone https://github.com/astronerd21/Deep-Learning-Oil-Slick-Detection.git
import sys
sys.path.append('/content/Deep-Learning-Oil-Slick-Detection')

In [ ]:
%%writefile /content/Deep-Learning-Oil-Slick-Detection/src/clean_dataset.py
import argparse
from pathlib import Path
import rasterio
import numpy as np

def check_nodata_ratio(filepath: Path, threshold: float = 0.5) -> bool:
    if not filepath.is_file(): return None
    try:
        with rasterio.open(filepath) as src:
            vv_band = src.read(1)
    except Exception as e:
        print(f"Warning: Skipping corrupt file {filepath}: {e}")
        return True
    nodata_pixels = np.sum((vv_band == 0.0) | (np.isnan(vv_band)) | (vv_band <= -160.0))
    total_pixels = vv_band.size
    return (nodata_pixels / total_pixels) > threshold

def clean_single_split_file(original_split_path, output_split_path, data_dir):
    if not original_split_path.is_file(): return
    valid_image_ids = []
    with open(original_split_path, "r") as f:
        image_ids = [line.strip() for line in f if line.strip()]
    for img_id in image_ids:
        img_file = img_id if img_id.endswith("_s1.tif") else f"{img_id.replace('.tif', '')}_s1.tif"
        if not check_nodata_ratio(data_dir / img_file, threshold=0.5):
            valid_image_ids.append(img_id)
    output_split_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_split_path, "w") as f:
        for img_id in sorted(valid_image_ids): f.write(f"{img_id}\n")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", required=True)
    parser.add_argument("--splits-in-dir", required=True)
    args = parser.parse_args()
    output_dir = Path("/content/cleaned_splits")
    clean_single_split_file(Path(args.splits_in_dir)/"train.txt", output_dir/"train_clean.txt", Path(args.data_dir))
    clean_single_split_file(Path(args.splits_in_dir)/"val.txt", output_dir/"val_clean.txt", Path(args.data_dir))

In [ ]:
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists('/content/data/images_s1'):
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-00.tar -C /content/data/
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-01.tar -C /content/data/

%cd /content/Deep-Learning-Oil-Slick-Detection
!PYTHONPATH=. python3 -m src.clean_dataset --data-dir "/content/data/images_s1" --splits-in-dir "/content/drive/MyDrive/OilSlickProject/data/OilSlick/splits/random"
!PYTHONPATH=. python3 -m src.evaluate --data-dir "/content/data/images_s1" --model "/content/drive/MyDrive/OilSlickProject/data/OilSlick/checkpoints/best_cleaned_model.pt" --split "/content/cleaned_splits/val_clean.txt" --backbone resnet18